# Библиотеки

In [7]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from docling.document_converter import DocumentConverter

from concurrent.futures import ThreadPoolExecutor
import html
import os
from pathlib import Path
import re
import requests
import time

# Извлечение текста

In [2]:
reports_dir = Path("data/reports")
files = []
for company in reports_dir.iterdir():
    for year in company.iterdir():
        for file in year.iterdir():
            files.append(file)


In [3]:
converter = DocumentConverter()

converted_documents = {}
for path in files:
    result = converter.convert(str(path))
    converted_documents[path] = result


W1118 13:37:09.986000 5924 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-11-18 13:38:06,367 MatchingPostProcessor WARNING  1 of 83 pdf cells matched neither a row nor a column band of the 22x7 grid and were dropped from the table
2026-11-18 13:43:42,113 MatchingPostProcessor WARNING  2 of 46 pdf cells matched neither a row nor a column band of the 5x7 grid and were dropped from the table
2026-11-18 13:48:14,203 MatchingPostProcessor WARNING  1 of 369 pdf cells matched neither a row nor a column band of the 18x12 grid and were dropped from the table


RapidOCR returned empty result!


2026-11-18 13:50:40,475 MatchingPostProcessor WARNING  1 of 177 pdf cells matched neither a row nor a column band of the 13x6 grid and were dropped from the table
2026-11-18 13:51:07,603 MatchingPostProcessor WARNING  1 of 158 pdf cells matched neither a row nor a column band of the 11x5 grid and were dropped from the table


RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-11-18 13:53:21,299 [RapidOCR] main.py:164: The text detection result is empty
RapidOCR returned empty result!


... (19 outputs left out)

RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR

2026-11-18 14:24:41,246 MatchingPostProcessor WARNING  1 of 400 pdf cells matched neither a row nor a column band of the 15x17 grid and were dropped from the table


RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-11-18 14:28:30,322 [RapidOCR] main.py:164: The text detection result is empty
RapidOCR returned empty result!


2026-11-18 14:31:09,423 MatchingPostProcessor WARNING  1 of 69 pdf cells matched neither a row nor a column band of the 3x7 grid and were dropped from the table


# Очистка

In [4]:
def clean_text(text):
    text = html.unescape(text)                                           # &gt; -> >
    text = re.sub(r"<!--.*?-->", " ", text)                              # <!-- image --> от Docling
    text = text.replace("\xa0", " ")                                     # неразрывные пробелы
    text = re.sub(r"^[ \t]*\d{1,3}[ \t]*(\n|$)", "", text, flags=re.M)   # строка из одного номера страницы
    text = re.sub(r"_{3,}", " ", text)                                   # пустые поля бланка
    text = re.sub(r"([а-яёa-z])-[ \t]*\n[ \t]*([а-яёa-z])", r"\1\2", text)  # перенос слова
    text = re.sub(r"[ \t]*\n[ \t]*(?=[а-яёa-z])", " ", text)             # разрыв строки посреди предложения
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [5]:
clean_documents = {}
for path, result in converted_documents.items():
    clean_documents[path] = []
    for i in range(len(result.document.pages)):
        clean_documents[path].append(clean_text(result.document.export_to_markdown(page_no=i + 1)))


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



# Чанкинг

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
all_chunks = []
for path in clean_documents.keys():
    for page_num, page in enumerate(clean_documents[path]):
        metadata = {
            "doc_id": f"{path.parent.parent.name}_{path.parent.name}_{path.name[:-4]}",
            "company": path.parent.parent.name,
            "year": path.parent.name,
            "file": path.name,
            "page": page_num + 1,
        }
        chunks = splitter.split_text(page)
        for chunk in chunks:
            all_chunks.append(Document(chunk, metadata=metadata))

# Векторная база

In [22]:
EMB_URL = "https://embeddings.parsers360.ru:10443"

class CourseEmbeddings(Embeddings):
    """Эмбеддер курса для LangChain: один текст — один запрос к сервису, в ответ вектор из 1024 чисел."""

    def __init__(self, api_key, url=EMB_URL, workers=4):
        self.headers = {"Authorization": f"Bearer {api_key}"}   # ключ передаём в заголовке запроса
        self.url = url
        self.workers = workers                                  # сколько запросов отправляем одновременно

    def embed_query(self, text):
        """Вектор для одного текста (вопроса или чанка)."""
        for attempt in range(5):                                # сервис перегружен — ждём и пробуем ещё раз
            response = requests.post(f"{self.url}/embed", headers=self.headers,
                                     json={"text": text[:20000]}, timeout=60)   # сервис принимает до 20 000 символов
            if response.status_code not in (429, 500, 502, 503, 504):
                break
            time.sleep(response.json()["Retry-After."])
        response.raise_for_status()                             # 401 — неверный ключ, остальные коды — ошибка сервиса
        return response.json()["embedding"]

    def embed_documents(self, texts):
        """Векторы для списка текстов: несколько запросов параллельно — так быстрее."""
        with ThreadPoolExecutor(self.workers) as pool:
            return list(pool.map(self.embed_query, texts))

def load_dotenv(path: Path = Path(".env")) -> None:
    """Читает .env. Переменные окружения важнее файла"""
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_dotenv()
EMB_KEY = os.environ.get("EMB_KEY")
embedder = CourseEmbeddings(EMB_KEY)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    persist_directory="data/embeddings",
    collection_metadata={"hnsw:space": "cosine"},
)